In [1]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(ggpubr)
    library(Matrix)
    library(SingleCellExperiment)
    library(Seurat)
    library(reshape2)
    library(scater)
    library(viridis)
})

options(repr.plot.width=15, repr.plot.height=8)

In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC'
io$output.directory <- file.path(io$basedir,"ArchR")
io$plot.dir = file.path(io$output.directory,'Plots')
setwd(io$output.directory)

In [3]:
addArchRThreads(threads = 1)

Setting default number of Parallel threads to 1.



In [10]:
io$archR.directory = file.path(io$output.directory, 'Project/')

ArchRProject.filt = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [5]:
addArchRChrPrefix(chrPrefix = TRUE)

ArchR is now requiring chromosome prefix = 'chr'



In [6]:
# seqlevels(ArchRProject.filt@geneAnnotation$TSS) = paste0('chr',seqnames(ArchRProject.filt@geneAnnotation$TSS)@values)
# seqlevels(ArchRProject.filt@geneAnnotation$exons) = paste0('chr',seqnames(ArchRProject.filt@geneAnnotation$exons)@values)
# seqlevels(ArchRProject.filt@geneAnnotation$genes) = paste0('chr',seqnames(ArchRProject.filt@geneAnnotation$genes)@values)
# seqlevels(ArchRProject.filt@genomeAnnotation$chromSizes) = paste0('chr',seqnames(ArchRProject.filt@genomeAnnotation$chromSizes)@values)

In [12]:
seqstyle = 'UCSC'
#seqstyle = 'NCBI'
seqlevelsStyle(ArchRProject.filt@genomeAnnotation$chromSizes) <- seqstyle
seqlevelsStyle(ArchRProject.filt@geneAnnotation$genes) <- seqstyle
seqlevelsStyle(ArchRProject.filt@geneAnnotation$TSS) <- seqstyle
seqlevelsStyle(ArchRProject.filt@geneAnnotation$exons) <- seqstyle

addArchRChrPrefix(chrPrefix = FALSE)

In [16]:
genome = getGenome( ArchRProject.filt)

In [17]:
# Create group coverage files
ArchRProject.filt <- addGroupCoverages(ArchRProj = ArchRProject.filt, groupBy = "sample")

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-3df5b8dbacff-Date-2021-12-03_Time-16-54-27.log
If there is an issue, please report to github with logFile!

rabbit_BGRGP1 (1 of 8) : CellGroups N = 2

rabbit_BGRGP2 (2 of 8) : CellGroups N = 2

rabbit_BGRGP3 (3 of 8) : CellGroups N = 2

rabbit_BGRGP4 (4 of 8) : CellGroups N = 2

rabbit_BGRGP5 (5 of 8) : CellGroups N = 2

rabbit_BGRGP6 (6 of 8) : CellGroups N = 2

rabbit_BGRGP7 (7 of 8) : CellGroups N = 2

rabbit_BGRGP8 (8 of 8) : CellGroups N = 2

2021-12-03 16:55:10 : Creating Coverage Files!, 0.723 mins elapsed.

2021-12-03 16:55:10 : Batch Execution w/ safelapply!, 0.723 mins elapsed.

2021-12-03 16:55:10 : Group rabbit_BGRGP1._.Rep1 (1 of 16) : Creating Group Coverage File : rabbit_BGRGP1._.Rep1.insertions.coverage.h5, 0.723 mins elapsed.

Number of Cells = 500

Coverage File Exists!

Added Coverage Group

Added Metadata Group

Added ArrowCoverage Class

Added Coverage/Info

Added Coverage/Info/CellNames

2021-12-03 16:56:06 : Gr

ERROR: Error in BSgenome[[availableChr[x]]]: no such sequence


In [13]:
pathToMacs2 <- findMacs2()

Searching For MACS2..

Found with $path!



In [14]:
# Find peaks
ArchRProject.filt <- addReproduciblePeakSet(
    ArchRProj = ArchRProject.filt, 
    groupBy = "celltype_manual", 
    pathToMacs2 = pathToMacs2
)

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-30e8d7e686a58-Date-2021-12-02_Time-16-06-06.log
If there is an issue, please report to github with logFile!

Calling Peaks with Macs2



ERROR: Error in .getCoverageMetadata(ArchRProj = ArchRProj, groupBy = groupBy, : No Coverage Metadata found for : celltype_manual. Please run addGroupCoverages!


In [ ]:
getPeakSet(ArchRProject.filt)

In [7]:
ArchRProject.filt@genomeAnnotation$chromSizes

GRanges object with 22 ranges and 0 metadata columns:
       seqnames      ranges strand
          <Rle>   <IRanges>  <Rle>
   [1]        1 1-194850757      *
   [2]       10  1-47997241      *
   [3]       11  1-87554214      *
   [4]       12 1-155355395      *
   [5]       13 1-143360832      *
   ...      ...         ...    ...
  [18]        6  1-27502587      *
  [19]        7 1-173684459      *
  [20]        8 1-111795807      *
  [21]        9 1-116251907      *
  [22]        X 1-111700775      *
  -------
  seqinfo: 22 sequences from an unspecified genome

In [124]:
devtools::install_github("GreenleafLab/chromVARmotifs")

Registered S3 method overwritten by 'cli':
  method     from         
  print.boxx spatstat.geom




glue    (1.5.0  -> 1.5.1 ) [CRAN]
memoise (2.0.0  -> 2.0.1 ) [CRAN]
stringi (1.7.5  -> 1.7.6 ) [CRAN]
cpp11   (0.4.1  -> 0.4.2 ) [CRAN]
vroom   (1.5.6  -> 1.5.7 ) [CRAN]
withr   (2.4.2  -> 2.4.3 ) [CRAN]
digest  (0.6.28 -> 0.6.29) [CRAN]
readr   (2.1.0  -> 2.1.1 ) [CRAN]


Skipping 28 packages ahead of CRAN: zlibbioc, IRanges, S4Vectors, BiocGenerics, DelayedArray, Biobase, MatrixGenerics, Rhtslib, BiocParallel, SummarizedExperiment, GenomeInfoDbData, BiocIO, GenomicAlignments, Rsamtools, Biostrings, GenomeInfoDb, XVector, GenomicRanges, AnnotationDbi, KEGGREST, GO.db, annotate, rtracklayer, seqLogo, DirichletMultinomial, CNEr, BSgenome, TFBSTools

Installing 8 packages: glue, memoise, stringi, cpp11, vroom, withr, digest, readr

Warning message in i.p(...):
“installation of package ‘glue’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



✔  checking for file ‘/tmp/RtmpxoTFL2/remotes30e8d26c739ac/GreenleafLab-chromVARmotifs-38bed55/DESCRIPTION’ (347ms)
─  preparing ‘chromVARmotifs’:
✔  checking DESCRIPTION meta-information
─  checking for LF line-endings in source and make files and shell scripts
─  checking for empty or unneeded directories
─  building ‘chromVARmotifs_0.2.0.tar.gz’
   


In [130]:
ArchRProject.filt = addReproduciblePeakSet(
  ArchRProj = ArchRProject.filt,
  groupBy = "celltype_manual",
  peakMethod = "Macs2")

Searching For MACS2..

Found with $path!

ArchR logging to : ArchRLogs/ArchR-addReproduciblePeakSet-30e8d2b966bbe-Date-2021-12-02_Time-17-40-13.log
If there is an issue, please report to github with logFile!

Calling Peaks with Macs2



ERROR: Error in .getCoverageMetadata(ArchRProj = ArchRProj, groupBy = groupBy, : No Coverage Metadata found for : celltype_manual. Please run addGroupCoverages!


In [128]:
# Motif enrichment
ArchRProject.filt <- addMotifAnnotations(ArchRProj = ArchRProject.filt, motifSet = "cisbp", name = "Motif", species='Homo sapiens')

ArchR logging to : ArchRLogs/ArchR-addMotifAnnotations-30e8d87e4135-Date-2021-12-02_Time-17-39-13.log
If there is an issue, please report to github with logFile!

2021-12-02 17:39:14 : Gettting Motif Set, Species : Homo sapiens, 0.016 mins elapsed.

Using version 2 motifs!

2021-12-02 17:39:17 : Finding Motif Positions with motifmatchr!, 0.069 mins elapsed.

peakSet is NULL. You need a peakset to run addMotifAnnotations! See addReproduciblePeakSet!



ERROR: Error: 



In [ ]:
markerTest <- getMarkerFeatures(
  ArchRProj = ArchRProject.filt, 
  useMatrix = "TileMatrix",
  groupBy = "celltype_manual",
  testMethod = "wilcoxon",
  bias = c("TSSEnrichment", "log10(nFrags)"))